In [ ]:
# %% [markdown]
# # Raw GDELT Source Check
#
# Purpose of this notebook:
# - inspect raw GDELT ZIP structure,
# - validate timestamp fields and fallback logic,
# - run a short raw-pipeline audit on a limited window.
#
# Important:
# - this is **not** the main baseline production route anymore,
# - the main baseline news panel is built via the final BigQuery SQL query,
# - this notebook is retained only as source-validation / audit evidence.

# %%
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

print("PROJECT_ROOT:", PROJECT_ROOT)

# %%
from src.paths import ROOT, RAW, INTERIM, PROCESSED

print("ROOT      :", ROOT)
print("RAW       :", RAW)
print("INTERIM   :", INTERIM)
print("PROCESSED :", PROCESSED)

# %% [markdown]
# ## 1. Inspect available raw ZIP files

# %%
import zipfile

news_folder = RAW / "news"
files = sorted(news_folder.glob("*.zip"))

print(f"Number of ZIP files found: {len(files)}")
print(files[:10])

# %%
if not files:
    raise FileNotFoundError(
        f"No ZIP files found in {news_folder}. "
        "Place a few raw GDELT news ZIPs there before running this notebook."
    )

zip_path = files[0]
print("Using sample ZIP:", zip_path.name)

# %%
with zipfile.ZipFile(zip_path, "r") as z:
    print("Archive contents:", z.namelist())

# %%
with zipfile.ZipFile(zip_path, "r") as z:
    with z.open(z.namelist()[0]) as f:
        for i in range(5):
            print(f"Line {i+1}:")
            print(f.readline().decode("utf-8", errors="replace"))

# %% [markdown]
# ## 2. Inspect raw column structure

# %%
import pandas as pd

sample = pd.read_csv(
    zip_path,
    sep="\t",
    header=None,
    compression="zip",
    nrows=5,
    dtype="string",
    on_bad_lines="skip",
)

print("Raw sample shape:", sample.shape)
sample.iloc[:, :8]

# %%
colnames = [
    "gkg_record_id",
    "date",
    "source_collection_id",
    "source_common_name",
    "document_identifier",
    "v1_counts",
    "v2_counts",
    "v1_themes",
    "v2_enhanced_themes",
]

sample_small = sample.iloc[:, :9].copy()
sample_small.columns = colnames

sample_small[["date", "source_common_name", "document_identifier", "v1_themes"]]

# %%
sample_small["v1_themes_split"] = sample_small["v1_themes"].str.split(";")
sample_small[["source_common_name", "v1_themes_split"]]

# %%
print("Full sample shape:", sample.shape)
sample.iloc[:, 24:27]

# %% [markdown]
# ## 3. Validate precise timestamp extraction from extras XML

# %%
sample_small["extras_xml"] = sample.iloc[:, 26]

sample_small["page_precise_pubtimestamp"] = sample_small["extras_xml"].str.extract(
    r"<PAGE_PRECISEPUBTIMESTAMP>(\d+)</PAGE_PRECISEPUBTIMESTAMP>"
)

sample_small[["source_common_name", "date", "page_precise_pubtimestamp"]]

# %%
sample_small["has_precise_timestamp"] = sample_small["page_precise_pubtimestamp"].notna()

sample_small[["source_common_name", "has_precise_timestamp", "page_precise_pubtimestamp"]]

# %%
sample_small["date_ymd"] = sample_small["date"].astype(str).str[:8]
sample_small["precise_ymd"] = sample_small["page_precise_pubtimestamp"].astype("string").str[:8]

sample_small["timestamp_mismatch"] = (
    sample_small["page_precise_pubtimestamp"].notna()
    & (sample_small["date_ymd"] != sample_small["precise_ymd"])
)

sample_small[[
    "source_common_name",
    "date_ymd",
    "precise_ymd",
    "timestamp_mismatch"
]]

# %%
sample_small["usable_timestamp"] = sample_small["page_precise_pubtimestamp"].astype("string")

mask = sample_small["page_precise_pubtimestamp"].isna() | sample_small["timestamp_mismatch"]
sample_small.loc[mask, "usable_timestamp"] = sample_small.loc[mask, "date"].astype("string")

sample_small[[
    "source_common_name",
    "date",
    "page_precise_pubtimestamp",
    "timestamp_mismatch",
    "usable_timestamp"
]]

# %% [markdown]
# ## 4. One-day raw download test (optional)
#
# This is just a small audit check.
# Skip this section if the ZIP files are already available locally.

# %%
from datetime import datetime, timedelta
from urllib.request import urlretrieve
from urllib.error import HTTPError, URLError

download_dir = RAW / "news"
download_dir.mkdir(parents=True, exist_ok=True)

# Small one-day test window
start = datetime(2017, 12, 5, 0, 0)
end = datetime(2017, 12, 5, 23, 45)

def iter_15min(start_dt, end_dt):
    current = start_dt
    while current <= end_dt:
        yield current
        current += timedelta(minutes=15)

downloaded = 0
already_exists = 0
missing = 0
errors = 0

for ts in iter_15min(start, end):
    stamp = ts.strftime("%Y%m%d%H%M00")
    url = f"http://data.gdeltproject.org/gdeltv2/{stamp}.gkg.csv.zip"
    dest = download_dir / f"{stamp}.gkg.csv.zip"

    if dest.exists():
        already_exists += 1
        continue

    try:
        urlretrieve(url, dest)
        downloaded += 1
    except HTTPError as e:
        if e.code == 404:
            missing += 1
        else:
            errors += 1
            print(f"HTTP error {e.code} for {stamp}")
    except URLError as e:
        errors += 1
        print(f"URL error for {stamp}: {e}")

print("downloaded     :", downloaded)
print("already_exists :", already_exists)
print("missing        :", missing)
print("errors         :", errors)

# %% [markdown]
# ## 5. Build a one-day raw frame and inspect fallback timestamp logic

# %%
zip_files = sorted(news_folder.glob("20171205*.gkg.csv.zip"))
print("Number of 2017-12-05 ZIP files found:", len(zip_files))

if not zip_files:
    print("No 20171205 files found. Skip this section if you did not run the optional download test above.")

# %%
frames = []

for zp in zip_files:
    try:
        df = pd.read_csv(
            zp,
            sep="\t",
            header=None,
            compression="zip",
            usecols=[0, 1, 3, 4, 7, 26],
            names=[
                "gkg_record_id",
                "date",
                "source_common_name",
                "document_identifier",
                "v1_themes",
                "extras_xml",
            ],
            dtype="string",
            on_bad_lines="skip",
        )

        df["page_precise_pubtimestamp"] = df["extras_xml"].str.extract(
            r"<PAGE_PRECISEPUBTIMESTAMP>(\d+)</PAGE_PRECISEPUBTIMESTAMP>"
        )

        df["date_ymd"] = df["date"].str[:8]
        df["precise_ymd"] = df["page_precise_pubtimestamp"].str[:8]

        df["timestamp_mismatch"] = (
            df["page_precise_pubtimestamp"].notna()
            & (df["date_ymd"] != df["precise_ymd"])
        )

        df["usable_timestamp"] = df["page_precise_pubtimestamp"]
        mask = df["page_precise_pubtimestamp"].isna() | df["timestamp_mismatch"]
        df.loc[mask, "usable_timestamp"] = df.loc[mask, "date"]

        frames.append(df)

    except Exception as e:
        print(f"Problem reading {zp.name}: {e}")

if frames:
    daily_gdelt = pd.concat(frames, ignore_index=True)

    daily_gdelt = daily_gdelt[
        [
            "gkg_record_id",
            "date",
            "page_precise_pubtimestamp",
            "usable_timestamp",
            "timestamp_mismatch",
            "source_common_name",
            "document_identifier",
            "v1_themes",
        ]
    ]

    print("daily_gdelt shape:", daily_gdelt.shape)
    display(daily_gdelt.head())
else:
    print("No frames were built for the one-day raw check.")

# %%
if "daily_gdelt" in globals():
    print("Timestamp mismatch counts:")
    print(daily_gdelt["timestamp_mismatch"].value_counts(dropna=False))
    print("\nShare with precise timestamp:")
    print(daily_gdelt["page_precise_pubtimestamp"].notna().mean())

# %% [markdown]
# ## 6. Short-window audit run using the backup article-level pipeline
#
# This is not the main production route.
# It is kept only to validate that the raw-source audit pipeline still works.

# %%
from src.paths import RAW, PROCESSED
from src.gdelt_article_audit import build_gdelt_news_dataset

# %%
daily_panel = build_gdelt_news_dataset(
    start_day="2017-12-01",
    end_day="2017-12-31",
    raw_news_dir=RAW / "news",
    processed_dir=PROCESSED,
    download_first=True,
    max_workers=12,
    save_article_level=True,
)

daily_panel.head()

# %% [markdown]
# ## 7. Interpretation
#
# Main takeaway from this notebook:
# - raw GDELT ZIP structure was validated,
# - precise page timestamps exist but are incomplete / noisy,
# - fallback day-level timestamp logic is defensible for raw-source audits,
# - raw article-level processing is feasible for short validation windows,
# - but the main baseline production route for the project is the final BigQuery daily aggregation query.